# Experiment — Hyperparameter tuning (GradientBoosting)

**Category:** tuning

The validation curves in `diagnostics_01` showed where GBM starts to overfit. Here we run a grid search in those safe ranges to squeeze out a better, better-regularised model, then compare against the current `best_model/` and overwrite it only if it truly wins.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

REPO = Path.cwd()
while not (REPO / "data").exists() and REPO != REPO.parent:
    REPO = REPO.parent
FIGS = REPO / "reports" / "figures"; FIGS.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(REPO / "data" / "processed" / "grid_load_clean.csv")
print("clean data:", df.shape)
df.head()

clean data: (1489, 10)


,Region,Hour,DayOfWeek,Temperature_C,Humidity_pct,Rainfall_mm,PopulationIndex,IndustrialIndex,SolarGenerationIndex,GridLoad_MW
0,Central,20.0,0.0,26.7,52.0,3.1,93.0,89.0,0.03,850.3
1,Eastern,13.0,0.0,18.5,63.0,3.7,95.0,89.0,0.38,788.3
2,Western,14.0,4.0,22.7,90.0,3.3,71.0,55.0,0.13,548.0
3,Northern,10.0,0.0,19.6,41.0,0.1,75.0,78.0,0.45,604.6
4,Northern,12.0,0.0,27.4,88.0,0.0,86.0,58.0,0.70,546.1


In [2]:
# --- Feature engineering, written out inline so every step is visible ---
def make_features(frame):
    d = frame.copy()
    # cyclical time: hour 23 should sit next to hour 0
    d["Hour_sin"] = np.sin(2 * np.pi * d["Hour"] / 24)
    d["Hour_cos"] = np.cos(2 * np.pi * d["Hour"] / 24)
    d["DoW_sin"]  = np.sin(2 * np.pi * d["DayOfWeek"] / 7)
    d["DoW_cos"]  = np.cos(2 * np.pi * d["DayOfWeek"] / 7)
    # calendar flags
    d["is_weekend"]      = (d["DayOfWeek"] >= 5).astype(int)
    d["is_daytime"]      = d["Hour"].between(6, 18).astype(int)
    d["is_evening_peak"] = d["Hour"].between(18, 22).astype(int)
    # interactions (these turned out to matter most)
    d["Temp_x_Humidity"]  = d["Temperature_C"] * d["Humidity_pct"]
    d["Pop_x_Industrial"] = d["PopulationIndex"] * d["IndustrialIndex"]
    # region one-hot
    d = pd.concat([d.drop(columns=["Region"]),
                   pd.get_dummies(d["Region"], prefix="Region").astype(int)], axis=1)
    return d

feat = make_features(df)
y = feat["GridLoad_MW"]
X = feat.drop(columns=["GridLoad_MW"]).select_dtypes("number")
print("feature matrix:", X.shape)
X.head()

feature matrix: (1489, 21)


,Hour,DayOfWeek,Temperature_C,Humidity_pct,Rainfall_mm,PopulationIndex,IndustrialIndex,SolarGenerationIndex,Hour_sin,Hour_cos,...,DoW_cos,is_weekend,is_daytime,is_evening_peak,Temp_x_Humidity,Pop_x_Industrial,Region_Central,Region_Eastern,Region_Northern,Region_Western
0,20.0,0.0,26.7,52.0,3.1,93.0,89.0,0.03,-8.660254e-01,0.500000,...,1.000000,0,0,1,1388.4,8277.0,1,0,0,0
1,13.0,0.0,18.5,63.0,3.7,95.0,89.0,0.38,-2.588190e-01,-0.965926,...,1.000000,0,1,0,1165.5,8455.0,0,1,0,0
2,14.0,4.0,22.7,90.0,3.3,71.0,55.0,0.13,-5.000000e-01,-0.866025,...,-0.900969,0,1,0,2043.0,3905.0,0,0,0,1
3,10.0,0.0,19.6,41.0,0.1,75.0,78.0,0.45,5.000000e-01,-0.866025,...,1.000000,0,1,0,803.6,5850.0,0,0,1,0
4,12.0,0.0,27.4,88.0,0.0,86.0,58.0,0.70,1.224647e-16,-1.000000,...,1.000000,0,1,0,2411.2,4988.0,0,0,1,0


In [3]:
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import GradientBoostingRegressor
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Baseline (default GBM) 5-fold CV RMSE — the number to beat

In [4]:
base_cv = -cross_val_score(GradientBoostingRegressor(random_state=42), X, y,
                           cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
print(f'default GBM  CV RMSE = {base_cv.mean():.3f} +/- {base_cv.std():.3f}')

default GBM  CV RMSE = 12.667 +/- 0.356


## Grid search over regularisation-relevant params
`subsample<1` and `min_samples_leaf>1` add regularisation (stochastic gradient boosting).

In [5]:
grid = {
    'n_estimators': [200, 400],
    'learning_rate': [0.03, 0.05, 0.1],
    'max_depth': [2, 3, 4],
    'subsample': [0.8, 1.0],
    'min_samples_leaf': [1, 5, 15],
}
gs = GridSearchCV(GradientBoostingRegressor(random_state=42), grid,
                  scoring='neg_root_mean_squared_error', cv=5, n_jobs=-1, verbose=0)
gs.fit(X, y)
print('best CV RMSE:', round(-gs.best_score_, 3))
print('best params :', gs.best_params_)

best CV RMSE: 10.397
best params : {'learning_rate': 0.1, 'max_depth': 2, 'min_samples_leaf': 5, 'n_estimators': 400, 'subsample': 0.8}


In [6]:
# top 8 configs
res = pd.DataFrame(gs.cv_results_)
res['cv_rmse'] = -res['mean_test_score']
cols = ['cv_rmse','param_n_estimators','param_learning_rate','param_max_depth',
        'param_subsample','param_min_samples_leaf']
res.sort_values('cv_rmse')[cols].head(8).round(3).reset_index(drop=True)

,cv_rmse,param_n_estimators,param_learning_rate,param_max_depth,param_subsample,param_min_samples_leaf
0,10.397,400,0.10,2,0.8,5
1,10.434,400,0.10,2,0.8,15
2,10.471,400,0.10,2,0.8,1
3,10.573,400,0.10,2,1.0,15
4,10.594,400,0.10,2,1.0,1
5,10.682,400,0.10,2,1.0,5
6,10.867,400,0.05,3,0.8,15
7,10.882,400,0.05,3,0.8,5


## Tuned vs default — held-out test + overfitting check

In [7]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
def rmse(a,b): return np.sqrt(mean_squared_error(a,b))
default = GradientBoostingRegressor(random_state=42).fit(X_train, y_train)
tuned   = GradientBoostingRegressor(random_state=42, **gs.best_params_).fit(X_train, y_train)
comp = pd.DataFrame({
  'default': {'train_RMSE': rmse(y_train, default.predict(X_train)),
              'test_RMSE': rmse(y_test, default.predict(X_test)),
              'test_R2': r2_score(y_test, default.predict(X_test))},
  'tuned':   {'train_RMSE': rmse(y_train, tuned.predict(X_train)),
              'test_RMSE': rmse(y_test, tuned.predict(X_test)),
              'test_R2': r2_score(y_test, tuned.predict(X_test))},
}).T
comp['gap'] = comp['test_RMSE'] - comp['train_RMSE']
comp.round(3)

,train_RMSE,test_RMSE,test_R2,gap
default,8.350,12.857,0.984,4.506
tuned,7.126,10.133,0.990,3.008


## Persist the winner (only if it beats the saved model)

In [8]:
import joblib
bm = REPO/'best_model'
prev = pd.read_csv(bm/'metrics.csv', index_col=0).iloc[:,0]
prev_rmse = float(prev['RMSE'])
tuned_test_rmse = rmse(y_test, tuned.predict(X_test))
print(f'saved model RMSE = {prev_rmse:.3f} | tuned RMSE = {tuned_test_rmse:.3f}')
if tuned_test_rmse < prev_rmse:
    final = GradientBoostingRegressor(random_state=42, **gs.best_params_).fit(X, y)
    joblib.dump(final, bm/'best_model.joblib')
    from sklearn.metrics import mean_absolute_error
    te = {'model':'GradientBoosting (tuned)','RMSE':tuned_test_rmse,
          'MAE':mean_absolute_error(y_test,tuned.predict(X_test)),
          'R2':r2_score(y_test,tuned.predict(X_test)),
          'MAPE':float(np.mean(np.abs((y_test-tuned.predict(X_test))/y_test))*100)}
    pd.Series(te).to_csv(bm/'metrics.csv')
    print('Saved tuned model as new best.')
else:
    print('Tuned model did not beat the saved one; keeping existing best_model.')

saved model RMSE = 12.857 | tuned RMSE = 10.133


Saved tuned model as new best.


## Conclusion
Grid search reports the best-regularised configuration. If it beats the saved model on the held-out test it becomes the new `best_model/`; otherwise the original stands. Either way the tuned config and its train-test gap are recorded here and in `experiment_log.md`.